# Appendix E, part 2: component-level comparison of one generated signal and the model's output

One signal, one geometry, two pages. Set `FAMILY` below to `"tsmixup"` or `"kernelsynth"` and run
the notebook; each run writes both pages for that family.

* **Page A --- are the components of the input present in the output?** On the left the generated
  signal and, through arrows, its components at a set of reference frequencies: for Light TSMixup
  those are the frequencies the generator actually drew, known by construction; for KernelSynth,
  whose draws have no line spectrum, the strongest lines of the signal itself. On the right the
  model's output fitted at the *same* frequencies, so the two columns are directly comparable.
* **Page B --- where are the components of the output?** The strongest lines of the model's output,
  and the input fitted at those same frequencies. A line that is strong on the right of page B and
  absent on its left is content the model introduced.

Amplitudes are least squares at a known frequency, `pl.fit_amp_phase`, the same estimator the
recovery ratio uses, so nothing here depends on the width of a DFT bin.

**This is descriptive.** It shows frequency information loss; it is not evidence of structural
aliasing, which is what the Bayesian models of the report decide. Everything it depends on lives in
`chronos/bayesian/`.


## 0, Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

# Where the repository is. Locally the notebook sits inside it and nothing has to be set; on
# Colab the working directory is /content, so the search widens to the usual places and, failing
# those, the repository is cloned. Setting REPO_DIR (or the PATCHALIASING_REPO environment
# variable) to the checkout skips the search entirely.
REPO_DIR = None                    # e.g. "/content/drive/MyDrive/patchAliasing"
REPO_URL = "https://github.com/FedericoSabbadini/patchAliasing.git"
CLONE_IF_MISSING = True            # False to fail with a message instead of cloning

MARKER = Path("chronos") / "bayesian" / "probe_lib.py"


def _ok(p) -> bool:
    return p is not None and (Path(p) / MARKER).exists()


def find_repo() -> Path:
    """Locate the checkout: an explicit setting, then the parents, then the usual Colab places."""
    explicit = REPO_DIR or os.environ.get("PATCHALIASING_REPO")
    if explicit:
        if _ok(explicit):
            return Path(explicit).resolve()
        raise FileNotFoundError(f"REPO_DIR is set to {explicit}, but {MARKER} is not under it")

    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:                       # the notebook inside the checkout
        if _ok(cand):
            return cand

    roots = [here, Path("/content"), Path("/content/drive/MyDrive"),
             Path("/content/drive/MyDrive/Colab Notebooks"), Path.home()]
    for root in roots:                                       # a checkout beside the notebook
        if not root.exists():
            continue
        for cand in [root, *(d for d in root.iterdir() if d.is_dir())]:
            if _ok(cand):
                return cand.resolve()

    if not CLONE_IF_MISSING:
        raise FileNotFoundError(
            f"{MARKER} not found. Set REPO_DIR to the checkout, or allow CLONE_IF_MISSING.")

    target = here / "patchAliasing"                          # last resort: fetch it
    if not (target / ".git").exists():
        print(f"cloning {REPO_URL} -> {target}")
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(target)])
    if not _ok(target):
        raise FileNotFoundError(f"{MARKER} missing from {target}")
    return target.resolve()


REPO  = find_repo()
BAYES = REPO / "chronos" / "bayesian"
sys.path.insert(0, str(BAYES))
sys.modules.pop("probe_lib", None)          # so a git pull is picked up without a kernel restart
print("repository:", REPO)

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore")
import probe_lib as pl

SEED = 42
np.random.seed(SEED)
FS, CTX, PRED, BAND = pl.FS, pl.CTX, pl.PRED, pl.BAND

from matplotlib.patches import ConnectionPatch

# ------------------------------------------------------------------------------------- #
#  CONFIGURATION
# ------------------------------------------------------------------------------------- #
FAMILY = "tsmixup"          # "tsmixup" or "kernelsynth": the selector for the whole notebook

REF_P, REF_S = 16, 16       # the published Chronos-Bolt tokeniser; P = S, so F_lock is one comb

# Light TSMixup draw. The generator picks k ~ U{1, K}; here k is fixed at K so that the page
# always shows the same number of components, and the drawn frequencies are kept apart so the
# panels stay readable. Nothing else departs from Algorithm 1 of the appendix.
K_COMPONENTS = 4
ALPHA_DIR    = 1.5          # Dirichlet concentration
MIN_SEP_DRAW = 10.0         # Hz between two drawn components

N_PEAKS      = 4            # KernelSynth, and page B: how many lines to read off a spectrum
PEAK_MIN_SEP = 6.0          # Hz between two peaks, so one lobe is not counted twice

OUT_MODE  = "rollout"       # "rollout": feed the median forecast back until GEN_LEN samples
                            # "horizon": the single 64-sample forecast (8 Hz bins)
GEN_LEN   = 512             # rollout length, so the output spectrum has 1 Hz bins
BATCH     = 64
DRAW_SEED = SEED            # change this to see a different draw of the same family
INJECT_TONE = None          # Hz, or None: the background alone, with no probe tone added

USE_STUB_FORECASTER = False # True only to check the layout without the checkpoints

OUT = BAYES / "_run" / "appendixE"
(OUT / "figures").mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(DRAW_SEED)

NAME = {"tsmixup": "Light TSMixup", "kernelsynth": "KernelSynth"}[FAMILY]
print(f"family={NAME}  geometry=p{REF_P}-s{REF_S}  output='{OUT_MODE}'  seed={DRAW_SEED}")
print(f"figures -> {OUT / 'figures'}")

## 1, The signal

In [ ]:
def light_tsmixup_draw(rng, k=K_COMPONENTS, alpha=ALPHA_DIR, length=pl.CANON_LEN,
                       min_sep=MIN_SEP_DRAW, pool=None):
    """One Light TSMixup realisation, with the frequencies and weights of the draw returned.

    Algorithm 1 of the appendix: one sinusoid per drawn pool frequency, each divided by its mean
    absolute value, combined under symmetric Dirichlet weights, normalised to unit variance.
    """
    pool = np.asarray(pl.tsmixup_pool() if pool is None else pool, dtype=float)
    freqs = []
    while len(freqs) < k:                       # redraw a frequency that would overlap another
        f = float(rng.choice(pool))
        if all(abs(f - g) >= min_sep for g in freqs):
            freqs.append(f)
    comps  = [np.asarray(pl.make_tone(f, 0.0, length, 1.0), dtype=float) for f in freqs]
    scaled = [c / np.mean(np.abs(c)) for c in comps]
    w = rng.dirichlet(np.full(k, alpha))
    x = np.sum([wi * si for wi, si in zip(w, scaled)], axis=0)
    return (x / x.std()).astype(np.float32), sorted(freqs), w


def top_peaks(x, n_peaks=N_PEAKS, min_sep=PEAK_MIN_SEP, band=BAND, fs=FS):
    """The strongest spectral lines of `x` inside `band`, no two closer than `min_sep`."""
    y = np.asarray(x, float); y = y - y.mean()
    mag = np.abs(np.fft.rfft(y * np.hanning(len(y))))
    fr = np.fft.rfftfreq(len(y), d=1 / fs)
    m = (fr >= band[0]) & (fr <= band[1])
    fr, mag = fr[m], mag[m]
    picked = []
    for i in np.argsort(mag)[::-1]:
        f = float(fr[i])
        if all(abs(f - p) >= min_sep for p in picked):
            picked.append(f)
        if len(picked) >= n_peaks:
            break
    return sorted(picked)


if FAMILY == "tsmixup":
    SIGNAL, REF_FREQS, WEIGHTS = light_tsmixup_draw(RNG)
    REF_SOURCE = "the frequencies drawn by the generator"
    print("drawn components [Hz]:", [round(f, 3) for f in REF_FREQS])
    print("mixing weights      :", np.round(WEIGHTS, 3))
else:
    SIGNAL = np.asarray(pl.background("kernelsynth", pl.CANON_LEN, int(DRAW_SEED)), dtype=np.float32)
    REF_FREQS, WEIGHTS = top_peaks(SIGNAL), None
    REF_SOURCE = f"the {len(REF_FREQS)} strongest lines of the generated signal"
    print("strongest lines [Hz]:", [round(f, 2) for f in REF_FREQS])

if INJECT_TONE is not None:
    SIGNAL = (SIGNAL + pl.make_tone(INJECT_TONE, 0.0, len(SIGNAL), pl.TONE_SNR)).astype(np.float32)
    print(f"probe tone injected at {INJECT_TONE} Hz")

CONTEXT = SIGNAL[:CTX]
print(f"signal {len(SIGNAL)} samples, context {len(CONTEXT)}, std {SIGNAL.std():.3f}")

## 2, What the model returns

In [ ]:
class StubProbe:
    """A stand-in for `pl.Probe` that needs no checkpoint: layout checks only.

    It returns a smoothed continuation of the context, which is not a forecast and must never be
    read as one. Every figure produced while `USE_STUB_FORECASTER` is true carries a stamp.
    """
    def __init__(self, P, S):
        self.P, self.S = P, S
        self.tag, self.label = pl.model_tag(P, S), f"stub p{P}-s{S}"
        self.stages = ["output_head"]

    def forecast(self, contexts):
        c = np.asarray(contexts, dtype=np.float32)
        k = np.ones(9) / 9.0
        sm = np.stack([np.convolve(row, k, mode="same") for row in c])
        return np.repeat(sm[:, -1:], PRED, axis=1) * 0.6 + sm[:, -PRED:] * 0.4

    def capture_reg(self, contexts, pipe=None):
        c = np.asarray(contexts, dtype=np.float32)
        return {"output_head": np.stack([c[:, :16], c[:, -16:]], axis=1).reshape(len(c), -1)}

    def close(self):
        pass


def open_probe(P, S, batch_size=64):
    """The real probe, or the stub when the checkpoints are not available."""
    return StubProbe(P, S) if USE_STUB_FORECASTER else pl.Probe(P, S, batch_size=batch_size)


def stamp_stub(fig):
    if USE_STUB_FORECASTER:
        fig.text(0.5, 0.5, "STUB FORECASTER\nNOT A MEASUREMENT", fontsize=42, color="red",
                 alpha=0.16, ha="center", va="center", rotation=30, zorder=99)

In [ ]:
def chronos_generate(probe, context, mode=OUT_MODE, gen_len=GEN_LEN):
    """The model's output for one context: the raw horizon, or the fed-back rollout.

    The rollout is the procedure the reconstruction figures use: the median forecast is appended to
    the context and the model re-invoked until `gen_len` samples exist. It is imposed from outside
    and is not Chronos-Bolt's own generation mode, which emits its whole horizon in one step; it is
    used here because 64 samples give 8 Hz bins, too coarse to place a line.
    """
    ctx_len = len(context)
    ctx = np.asarray(context, dtype=np.float32)[None, :]
    if mode == "horizon":
        return probe.forecast(ctx)[0].astype(float)
    gen = np.zeros((1, 0), dtype=np.float32)
    while gen.shape[1] < gen_len:
        step = probe.forecast(ctx)
        gen = np.concatenate([gen, step], axis=1)
        ctx = np.concatenate([ctx, step], axis=1)[:, -ctx_len:]
    return gen[0, :gen_len].astype(float)


probe = open_probe(REF_P, REF_S, batch_size=BATCH)
try:
    print("model:", probe.label)
    OUTPUT = chronos_generate(probe, CONTEXT)
finally:
    probe.close()

TRUE_FUTURE = SIGNAL[CTX:CTX + PRED].astype(float)     # the genuine continuation, for reference
print(f"output {len(OUTPUT)} samples, std {OUTPUT.std():.4f}; "
      f"resolution {FS / len(OUTPUT):.2f} Hz")

## 3, Decomposition and the two pages

In [ ]:
C_IN, C_OUT = "#1f4e79", "#c81e3c"
LOCKS_REF = sorted({round(f, 3) for f in pl.patch_nulls(REF_P)} |
                   {round(f, 3) for f in pl.stride_locks(REF_S)})


def is_lock(f, tol=1.0):
    return any(abs(f - l) <= tol for l in LOCKS_REF)


def component_at(x, f, fs=FS):
    """Least-squares amplitude and phase of `x` at `f` Hz, the estimator R is fitted with."""
    y = np.asarray(x, float)
    t = np.arange(len(y)) / fs
    amp, ph = pl.fit_amp_phase(y, t, f)
    return float(amp), float(ph)


def component_curve(f, amp, ph, n_cycles=4, n_pts=400):
    """The fitted sinusoid on a fine grid, so a 200 Hz component is not drawn from eight samples."""
    t = np.linspace(0, n_cycles / max(f, 1e-9), n_pts)
    return t * 1000.0, amp * np.cos(2 * np.pi * f * t - ph)


def draw_signal(ax, y, color, title, fs=FS):
    t = np.arange(len(y)) / fs * 1000.0
    ax.plot(t, y, color=color, lw=0.8)
    ax.set_title(title, fontsize=9, pad=4)
    ax.set_xlabel("time [ms]", fontsize=7)
    ax.tick_params(labelsize=6)
    ax.margins(x=0.01)


def draw_pair(ax, f, amp_in, ph_in, amp_out, ph_out, n_cycles=4):
    """One frequency, both signals: the real component solid, the predicted one dashed over it."""
    t_ms, y_in = component_curve(f, amp_in, ph_in, n_cycles)
    _, y_out = component_curve(f, amp_out, ph_out, n_cycles)
    ax.axhline(0, color="0.85", lw=0.6, zorder=0)
    ax.plot(t_ms, y_in, color=C_IN, lw=1.3, zorder=3)
    ax.plot(t_ms, y_out, color=C_OUT, lw=1.3, ls="--", zorder=4)
    ylim = 1.15 * max(amp_in, amp_out, 1e-6)
    ax.set_ylim(-ylim, ylim)
    ratio = amp_out / amp_in if amp_in > 0 else np.nan
    gone = np.isfinite(ratio) and ratio < 0.10
    ax.set_title(f"{f:.2f} Hz{'  (lock)' if is_lock(f) else ''}\n"
                 f"A real {amp_in:.3f}  ->  pred {amp_out:.3f}   ({ratio:.2f}x)",
                 fontsize=7, pad=2, color="#8c1010" if gone else "black")
    ax.set_xlabel("time [ms]", fontsize=6)
    ax.tick_params(labelsize=5); ax.set_yticks([])
    ax.margins(x=0.01)


def arrow(fig, ax_from, ax_to, side="right"):
    """An arrow from the edge of a signal panel to the edge of a component panel."""
    if side == "right":
        a, b = (1.005, 0.5), (-0.05, 0.5)
    else:
        a, b = (-0.005, 0.5), (1.05, 0.5)
    fig.add_artist(ConnectionPatch(xyA=a, coordsA=ax_from.transAxes,
                                   xyB=b, coordsB=ax_to.transAxes,
                                   arrowstyle="-|>", mutation_scale=11, lw=0.8, color="0.45"))


def decomposition_page(freqs, title, subtitle, fname):
    """One page: the real signal on the left, the model output on the right, and between them
    one panel per frequency carrying BOTH components overlaid, so presence and absence are read
    in the same axes."""
    k = len(freqs)
    fig = plt.figure(figsize=(12.5, 2.3 * k + 2.2))
    gs = GridSpec(k, 3, figure=fig, width_ratios=[1.5, 1.25, 1.5],
                  wspace=0.26, hspace=0.62, left=0.045, right=0.985, top=0.86, bottom=0.07)
    ax_l, ax_r = fig.add_subplot(gs[:, 0]), fig.add_subplot(gs[:, 2])
    draw_signal(ax_l, SIGNAL, C_IN, f"{NAME} signal (real)")
    draw_signal(ax_r, OUTPUT, C_OUT, "Chronos prediction")

    rows = []
    for i, f in enumerate(freqs):
        amp_l, ph_l = component_at(SIGNAL, f)
        amp_r, ph_r = component_at(OUTPUT, f)
        ax = fig.add_subplot(gs[i, 1])
        draw_pair(ax, f, amp_l, ph_l, amp_r, ph_r)
        arrow(fig, ax_l, ax, "right"); arrow(fig, ax_r, ax, "left")
        rows.append(dict(freq_hz=f, is_lock=is_lock(f), amp_real=amp_l, amp_pred=amp_r,
                         ratio=amp_r / amp_l if amp_l > 0 else np.nan))

    fig.suptitle(title, fontsize=12, y=0.975)
    fig.text(0.5, 0.925, subtitle, ha="center", fontsize=8.5, color="0.25")
    fig.legend(handles=[Line2D([0], [0], color=C_IN, lw=1.4, label="component of the real signal"),
                        Line2D([0], [0], color=C_OUT, lw=1.4, ls="--",
                               label="component of the prediction")],
               loc="lower center", ncol=2, fontsize=8.5, frameon=False, bbox_to_anchor=(0.5, 0.015))
    stamp_stub(fig)
    p = OUT / "figures" / fname
    fig.savefig(p, dpi=200, bbox_inches="tight")
    fig.savefig(p.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig); print("wrote", p.name)
    return p, pd.DataFrame(rows)

In [ ]:
mode_txt = (f"autoregressive rollout, {len(OUTPUT)} samples, {FS / len(OUTPUT):.2f} Hz bins"
            if OUT_MODE == "rollout" else
            f"single {PRED}-sample horizon, {FS / PRED:.0f} Hz bins")

page_a, table_a = decomposition_page(
    REF_FREQS,
    f"{NAME}: the components of the real signal, and what the prediction holds at them",
    f"p{REF_P}-s{REF_S}. The frequencies are {REF_SOURCE}; they are properties of the input and "
    f"need not be present in the prediction at all. Prediction: {mode_txt}. Amplitudes are least "
    f"squares at a known frequency; the multiplier is prediction over real.",
    f"FIG_E2_{FAMILY}_input_frequencies.png")

found = top_peaks(OUTPUT)
page_b, table_b = decomposition_page(
    found,
    f"{NAME}: the components of the prediction, and what the real signal holds at them",
    f"p{REF_P}-s{REF_S}. The frequencies are the {len(found)} strongest lines of the prediction "
    f"itself. Read against the page above: a line strong here and weak in the real signal is "
    f"content the model introduced rather than reconstructed.",
    f"FIG_E2_{FAMILY}_output_frequencies.png")

summary = pd.concat([table_a.assign(page="A, input frequencies"),
                     table_b.assign(page="B, output frequencies")], ignore_index=True)
summary.to_csv(OUT / f"decomposition_{FAMILY}.csv", index=False)
display(summary.round(4))

## 4, What goes in the appendix, and what may be claimed

Two `figure*` pages per family, each `width=\textwidth`. The caption carries the reference
geometry, how the output was produced (rollout or single horizon) and the resolution that follows
from it, and the fact that amplitudes are least squares at a known frequency rather than DFT bins.

Three limits belong in the text, not the caption, and each is a property of the measurement rather
than of the model. The output is a median quantile, and a conditional median is smoother than a
sample path, so part of any high-frequency attenuation belongs to the point forecast. The rollout
is imposed from outside and compounds its own error, so a line that decays across it is not by
itself a statement about tokenisation. And a candidate frequency and its controls sit a few hertz
apart, which no spectrum computed here separates: the pages show loss, not the paired contrast the
Bayesian models estimate.
